# Recursive subagents and scope

In tutorial 1 every input was a variable in *one* agent's REPL. But an agent can call `invoke` itself, making a tree of agents — which is how you split a job too big for one REPL session.

That raises the obvious question: what can a **subagent** see? The answer is the point of this tutorial, and it is not “everything”.

## Setup

Install JAZ, provide an API key, and pick a model. Everything below runs in Colab — no local install.

In [1]:
%pip install -q jaz-lang

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

In [3]:
import jaz
from jaz.llm import LiteLLM

jaz.configure(llm=LiteLLM(model='openai/gpt-5.4-nano'))

## An agent calling agents

`invoke` is already available inside the REPL — nothing to import or pass in. Here the top-level agent farms each ticket out to its own subagent.

In [4]:
from jaz import invoke

categories = invoke(
    task=(
        "For each ticket in tickets, call invoke(task=..., ticket=...) to get a one-word "
        "category for it. Collect the results and return the list."
    ),
    tickets=[
        "Card declined at checkout",
        "How do I export my data?",
        "App crashes on launch",
    ],
)

categories

['Payment', 'Data', 'Bug']

## Inputs stop at the agent that received them

Here is the part that surprises people. An input you pass to `invoke` is bound in **that agent's** REPL and nowhere else. Subagents do not inherit it.

We check it by reading the prompts rather than asking an agent to tell us — the prompt is ground truth. (Same scaffolding as last time; skip the class.)

In [5]:
from jaz.hooks import Hook


class CollectPrompts(Hook):
    """Records the prompt of every invoke in the tree.

    Same scaffolding as tutorial 1's PrintUserPrompt, just keeping all of them instead of
    printing the first. You do not need to understand it; hooks are tutorial 3.
    """

    def __init__(self):
        self.prompts = []

    def on_llm_query_enter(self, event):
        whole = "\n".join(m["content"] for m in event.messages)
        if whole not in self.prompts:
            self.prompts.append(whole)
        return []


def stock_level(sku: str) -> int:
    """Units currently in stock for a SKU."""
    return {"SKU-1": 12, "SKU-2": 0}[sku]


TASK = "Call invoke(task='Return the number 1.') and return its result."

In [6]:
seen = CollectPrompts()

with seen:
    invoke(task=TASK, stock_level=stock_level)

for i, prompt in enumerate(seen.prompts):
    print(f"invoke #{i}  can see stock_level: {'stock_level' in prompt}")

invoke #0  can see stock_level: True
invoke #1  can see stock_level: False


The top-level agent can see `stock_level`. The subagent it spawned cannot — the name is not bound down there at all.

That is deliberate: an agent's inputs are its own, and a deep tree does not silently accumulate every caller's variables. But it does mean that passing a tool the ordinary way gives it to exactly one agent.

## `scope` — inputs for a whole subtree

`scope` binds values as **ambient inputs**: everything inside the `with` block sees them, nested invokes included, all the way down.

Same code, one line different:

In [7]:
from jaz import scope

seen = CollectPrompts()

with seen, scope(stock_level=stock_level):
    invoke(task=TASK)

for i, prompt in enumerate(seen.prompts):
    print(f"invoke #{i}  can see stock_level: {'stock_level' in prompt}")

invoke #0  can see stock_level: True
invoke #1  can see stock_level: True


Both can see it now, and nothing else changed.

This is the usual way to hand an agent its tools: most tools should be available to the whole tree rather than to the top call alone. It is equivalent to passing the value to every `invoke` beneath it by hand, without the hand-threading.

(A detail you may notice if you print the prompts: a per-call input is described in the **user** message, a scoped value in the **system** message — it is ambient, so it belongs to the session rather than to one request.)

Now a real use — subagents that actually call the scoped tool:

In [8]:
with scope(stock_level=stock_level):
    stock = invoke(
        task=(
            "For each of SKU-1 and SKU-2, call invoke(task=...) and have the subagent call "
            "stock_level to look the value up. Return a flat dict mapping each sku string "
            "to its integer stock level."
        ),
    )

stock

{'SKU-1': 12, 'SKU-2': 0}

## What to take away

- **A subagent is just `invoke` called from inside the REPL.**
- **Inputs do not propagate.** They belong to the agent that received them.
- **`scope` reaches the subtree.** Bind tools and shared data there, not as per-call inputs.

Recursion is unbounded by default. Capping it — like everything else that shapes a run — is done with a hook, which is next.

---

**Next:** [Hooks, events and effects](03-hooks-events-and-effects.ipynb) — including what those `Hook` classes you have been skipping actually do.